In [ ]:
from bs4 import BeautifulSoup
import urllib.request
import pandas as pd
import re
import os


In [ ]:
html_doc = source = urllib.request.urlopen('https://www.newsguardtech.com/coronavirus-misinformation-tracking-center/').read()
soup = BeautifulSoup(html_doc, 'html.parser')

In [ ]:
fakelist = [x.string.lower() for x in soup.find_all('a') if x.string is not None and len(re.findall('\\.',x.string)) > 0][1:]
#re.findall('.*\\.',x.string)[0][:-1]
print(len(fakelist))
fakelist

In [ ]:
cwd = os.getcwd()
cwd

In [ ]:
subm_files = sorted(re.findall('SUBM_2020_[0-9][0-6].csv', ' '.join(os.listdir(cwd+'/../output'))))
comm_files = sorted(re.findall('COMM_2020_[0-9][0-6].csv', ' '.join(os.listdir(cwd+'/../output'))))
match_files = sorted(re.findall('MATCH_2020_[0-9][0-6].csv', ' '.join(os.listdir(cwd+'/../output'))))

In [ ]:
SS = pd.DataFrame()
for ss,cc,mm in zip(subm_files, comm_files, match_files):
    SUBM = pd.read_csv(cwd+'/../output/'+ss, lineterminator = '\n')
    COMM = pd.read_csv(cwd+'/../output/'+cc, lineterminator = '\n')
    MATCH = pd.read_csv(cwd+'/../output/'+mm, lineterminator = '\n')
    SUBM['was_removed'] = (~SUBM.removed_by_category.isna()).astype(int)
    FLAGS = COMM.rename(columns={'id':'comm_id','parent_id':'subm_id'})\
                    [['subm_id','comm_id']].set_index('comm_id').\
                    join(MATCH[['comm_id','flag']].\
                    set_index('comm_id')).fillna(0).\
                    groupby('subm_id')['flag'].agg('sum')
    SS0 = SUBM.rename(columns={'id':'subm_id'})[['subm_id','subreddit','domain','was_removed']].\
                set_index('subm_id').join(FLAGS, how = 'left').fillna(0)
    SS=SS.append(SS0)
    print('Finished: ' + ss)

In [ ]:
SS['domain'] = SS.domain.astype('str')
SS['isflagged'] = (SS.flag>0).astype('int')
SS['fakedomain'] = (SS.domain.isin(fakelist)).astype('int')
SS['total']=1
SS

In [ ]:
#SS.domain.sample(20)
SSD = SS[~SS.domain.str.contains('redd\\.it|^self\\.|twitter|/r/|youtu|img|gif|giph|blog')]
SSD.reset_index().to_csv("SSD.csv")
SSD

In [ ]:
pdc = pd.crosstab(SSD.isflagged, SSD.fakedomain)
pdc

In [ ]:
round(100*pdc/pdc.agg('sum'),3)

In [ ]:
import numpy as np
round(100*(pdc.transpose()/pdc.transpose().agg('sum')).transpose(),3)

### Aggregate at domain level

In [ ]:
SSD = pd.read_csv("SSD.csv")

In [ ]:
SSD.domain = SSD.domain.astype('str')

In [ ]:
DOMAGG = SSD[SSD.fakedomain==0].groupby('domain')[['isflagged','total']].agg('sum')
DOMAGG['pct'] = round(100*(DOMAGG.isflagged/DOMAGG.total),2)
DOMAGG

Fake news websites

In [ ]:
pd.set_option('display.max_rows', 200)
DOMAGGF = SSD[SSD.fakedomain>0].groupby('domain')[['isflagged','total']].agg('sum')
DOMAGGF['pct'] = round(100*(DOMAGGF.isflagged/DOMAGGF.total),2)
print(DOMAGGF.sort_values('total', ascending = False))
pd.set_option('display.max_rows', 30)

In [ ]:
fakelist_mins = list(DOMAGGF[DOMAGGF.total>=20].index)
nonfakelist_mins = list(DOMAGG[DOMAGG.total>=20].index)

In [ ]:
print('%7.0d / %7.0d (%5.2f%%) fake news domains present on Reddit.' %
      (len(set(SSD.domain.unique()).intersection(set(fakelist))),
       len(set(fakelist)),
       100*len(set(SSD.domain.unique()).intersection(set(fakelist)))/len(set(fakelist))))

print('\n%7.0d / %7.0d (%5.2f%%) fake news domains flagged on Reddit.' %
      (len(set(SSD.domain[SSD.isflagged>0].unique()).intersection(set(fakelist))),
       len(set(SSD.domain.unique()).intersection(set(fakelist))),
       100*len(set(SSD.domain[SSD.isflagged>0].unique()).intersection(set(fakelist))) / len(set(SSD.domain.unique()).intersection(set(fakelist)))))

print('%7.0d / %7.0d (%5.2f%%) non-fake news domains flagged on Reddit.' %
      (len(set(SSD.domain[SSD.isflagged>0].unique()).difference(set(fakelist))),
       len(set(SSD.domain.unique()).difference(set(fakelist))),
       100*len(set(SSD.domain[SSD.isflagged>0].unique()).difference(set(fakelist))) / len(set(SSD.domain.unique()).difference(set(fakelist)))))

print('\n%7.0d / %7.0d (%5.2f%%) fake news domains (with at least 5 submissions) flagged on Reddit.' %
      (len(set(SSD.domain[SSD.isflagged>0].unique()).intersection(set(fakelist_mins))),
       len(set(SSD.domain.unique()).intersection(set(fakelist_mins))),
       100*len(set(SSD.domain[SSD.isflagged>0].unique()).intersection(set(fakelist_mins))) / len(set(SSD.domain.unique()).intersection(set(fakelist_mins)))))

print('%7.0d / %7.0d (%5.2f%%) non-fake news domains (with at least 5 submissions) flagged on Reddit.' %
      (len(set(SSD.domain[SSD.isflagged>0].unique()).intersection(set(nonfakelist_mins))),
       len(set(SSD.domain.unique()).intersection(set(nonfakelist_mins))),
       100*len(set(SSD.domain[SSD.isflagged>0].unique()).intersection(set(nonfakelist_mins))) / len(set(SSD.domain.unique()).intersection(set(nonfakelist_mins)))))

In [ ]:
DOMAGGF[DOMAGGF.pct>0].pct.mean()

In [ ]:
DOMAGG[DOMAGG.pct>0].pct.mean()

In [ ]:
SSDSUBR = SSD[SSD.fakedomain>0].groupby(['subreddit','domain']).\
                agg(totalposts = ('total','sum'), totalflagged = ('isflagged','sum')).\
                assign(ispresent = lambda df: (df.totalposts>0).astype('int'),
                       wasflagged = lambda df: (df.totalflagged>0).astype('int'))
SSDSUBR.head()

### Which subreddits post and flag dubious sources

In [ ]:
pd.set_option('display.max_rows', 200)
print(SSDSUBR.reset_index().\
        groupby('subreddit')[['ispresent','wasflagged']].agg('sum').\
        assign(percflagged = lambda df: round(100*df.wasflagged/df.ispresent,2)))
pd.set_option('display.max_rows', 30)

### Do flagged comments get removed?

In [ ]:
pd.crosstab(SSD.was_removed, SSD.isflagged)

In [ ]:
pd.crosstab(SSD.was_removed, SSD.isflagged).apply(lambda r: r/r.sum(), axis=0)

In [ ]:
SSD['fndom'] = SSD.domain.isin(fakelist).astype(int)
SSD.groupby('fndom')['was_removed'].agg(['mean'])

In [ ]:
SSDOM2 = SSD.groupby('domain')['was_removed'].agg(['sum','count']).reset_index()
SSDOM2['percremoved'] = SSDOM2['sum']/SSDOM2['count']
SSDOM2['fndom'] = SSDOM2.domain.isin(fakelist).astype(int)

In [ ]:
SSDOM2.groupby('fndom')['percremoved'].agg(['min','max','mean','median'])